# BME i9400 — Meeting 4
## Linear Algebra for Machine Learning

**Fall 2026 · Monday, September 14**

Today's lecture is about representing data as vectors and matrices, and about what can be measured
once it is represented that way: distance, similarity, and the directions along which the data
actually varies.

**Agenda**

1. Data as a matrix
2. Vectors: length, distance, and the dot product
3. Projection
4. Covariance
5. Principal components
6. The singular value decomposition
7. Embeddings
8. Connection to the course

---
## 1 — Data as a matrix

Every dataset in this course is a table of numbers, written $X$.

$$X \in \mathbb{R}^{n \times d}$$

> $\mathbb{R}^{n \times d}$ means "the set of tables of real numbers with $n$ rows and $d$ columns".
>
> $X \in \mathbb{R}^{n \times d}$ is read "$X$ is an $n$-by-$d$ table of real numbers".

- $n$ is the number of **samples** — one per row
- $d$ is the number of **features** — one per column

For the mammography data of Meeting 2, a row was one woman and a column was one measurement. For
today's example:

- $n = 801$ tumour samples
- $d = 400$ genes, each column holding that gene's expression level

Two conventions worth fixing now, because they are the source of most shape errors you will hit this
semester:

- **Rows are samples, columns are features.** Always, in this course and in scikit-learn.
- $x_i$ denotes **row $i$** — one sample, a vector of $d$ numbers.

A single row is a point in $d$-dimensional space. A dataset is a cloud of $n$ such points. Almost
everything that follows is a statement about the shape of that cloud.

---
## 2 — Vectors: length, distance, and the dot product

A row $x = (x_1, x_2, \dots, x_d)$ is a **vector**: an ordered list of $d$ numbers, which we treat as
a point, or as an arrow from the origin to that point.

### Length

$$\|x\| = \sqrt{x_1^2 + x_2^2 + \cdots + x_d^2} = \sqrt{\sum_{j=1}^{d} x_j^2}$$

> $\|x\|$ is the **norm**, or length, of $x$. In two dimensions this is Pythagoras' theorem; the
> formula above is the same statement in $d$ dimensions.

### Distance between two samples

$$\|x - z\| = \sqrt{\sum_{j=1}^{d} (x_j - z_j)^2}$$

The length of the difference vector. Two patients with similar measurements are close together in
this sense, and "close together" is the entire basis of nearest-neighbour methods.

### The dot product

$$x \cdot z = x_1 z_1 + x_2 z_2 + \cdots + x_d z_d = \sum_{j=1}^{d} x_j z_j$$

> The **dot product** multiplies the two vectors element by element and adds up the results. It takes
> two vectors and returns a single number.

Two facts make the dot product the most used operation in machine learning:

- $x \cdot x = \|x\|^2$. A vector dotted with itself gives its squared length.
- $x \cdot z = \|x\|\,\|z\| \cos\theta$, where $\theta$ is the angle between the two vectors.

The second identity means the dot product measures **alignment**. Rearranging it gives a similarity
score that ignores magnitude:

$$\cos\theta = \frac{x \cdot z}{\|x\|\,\|z\|}$$

> This is called **cosine similarity**. It equals $1$ when two vectors point the same way, $0$ when
> they are perpendicular, and $-1$ when they point in opposite directions.

Cosine similarity is used when the *pattern* of a sample matters more than its overall scale — two
tumour samples with the same relative gene expression profile but different total RNA yield should
count as similar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.rcParams.update({"font.size": 13, "figure.dpi": 110})

# Two patients, described by two measurements each, so we can draw them.
a = np.array([2.0, 5.0])      # patient A: [glucose (scaled), BMI (scaled)]
b = np.array([5.0, 3.0])      # patient B

dist   = np.linalg.norm(a - b)
dot    = a @ b                                    # @ is the dot product in NumPy
cosine = dot / (np.linalg.norm(a) * np.linalg.norm(b))
angle  = np.degrees(np.arccos(cosine))

print(f"||a||            = {np.linalg.norm(a):.3f}")
print(f"||b||            = {np.linalg.norm(b):.3f}")
print(f"||a - b||        = {dist:.3f}      (distance between the patients)")
print(f"a . b            = {dot:.3f}")
print(f"cosine similarity= {cosine:.3f}    (angle = {angle:.1f} degrees)")

fig, ax = plt.subplots(figsize=(6.5, 6))
for v, name, col in [(a, "a", "#c1272d"), (b, "b", "#2b6cb0")]:
    ax.annotate("", xy=v, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", lw=2.5, color=col))
    ax.text(v[0]*1.06, v[1]*1.06, name, color=col, fontsize=17, weight="bold")
ax.plot([a[0], b[0]], [a[1], b[1]], "--", color="#6e6878", lw=1.8)
ax.text((a[0]+b[0])/2 + 0.15, (a[1]+b[1])/2 + 0.15, f"$\\|a-b\\|$ = {dist:.2f}", color="#6e6878")
ax.set_xlim(0, 6.5); ax.set_ylim(0, 6.5); ax.set_aspect("equal")
ax.set_xlabel("feature 1"); ax.set_ylabel("feature 2")
ax.set_title(f"Two samples as vectors  (angle {angle:.0f}°)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 3 — Projection

Projection answers: **how much of vector $x$ lies along direction $u$?**

Take $u$ to be a **unit vector**, meaning $\|u\| = 1$.

> A **unit vector** has length exactly 1. Any nonzero vector can be made into one by dividing it by
> its own length: $u = v / \|v\|$. A unit vector carries a direction and no magnitude.

The amount of $x$ lying along $u$ is a single number, the dot product:

$$\text{coordinate of } x \text{ along } u \;=\; x \cdot u$$

and the projected vector itself — the point on the line through $u$ closest to $x$ — is

$$\text{proj}_u(x) = (x \cdot u)\,u$$

Deriving it in two steps:

- write the projection as some multiple of $u$, say $c\,u$, since it must lie on the line
- choose $c$ so that the leftover $x - c\,u$ is perpendicular to $u$, which requires
  $(x - c\,u) \cdot u = 0$, hence $x \cdot u - c\,(u \cdot u) = 0$, and since $u \cdot u = \|u\|^2 = 1$,
  we get $c = x \cdot u$

Projection is how dimensionality reduction works. Replacing $x$ by the single number $x \cdot u$
takes a $d$-dimensional sample down to one dimension. Doing it with two well-chosen directions gives
the two-dimensional plots in section 5. **The whole question is which $u$ to choose**, and section 5
answers it.

In [ ]:
x = np.array([4.0, 3.0])
v = np.array([3.0, 1.0])
u = v / np.linalg.norm(v)            # unit vector: direction only

c    = x @ u                          # the coordinate of x along u
proj = c * u                          # the projected vector
residual = x - proj                   # what projection discards

print(f"u             = [{u[0]:.3f}, {u[1]:.3f}]   (length {np.linalg.norm(u):.3f})")
print(f"x . u         = {c:.3f}       <- x compressed to a single number")
print(f"proj_u(x)     = [{proj[0]:.3f}, {proj[1]:.3f}]")
print(f"residual      = [{residual[0]:.3f}, {residual[1]:.3f}]")
print(f"residual . u  = {residual @ u:.2e}   <- zero: the leftover is perpendicular to u")

fig, ax = plt.subplots(figsize=(7, 6))
lim = 5.5
ax.plot([-lim*u[0], lim*u[0]], [-lim*u[1], lim*u[1]], color="#b8b2c4", lw=1.5, zorder=1)
ax.annotate("", xy=x,    xytext=(0,0), arrowprops=dict(arrowstyle="-|>", lw=2.5, color="#c1272d"))
ax.annotate("", xy=proj, xytext=(0,0), arrowprops=dict(arrowstyle="-|>", lw=2.5, color="#2b6cb0"))
ax.plot([x[0], proj[0]], [x[1], proj[1]], ":", color="#6e6878", lw=2)
ax.text(x[0]+0.12, x[1]+0.12, "$x$", color="#c1272d", fontsize=17, weight="bold")
ax.text(proj[0]+0.12, proj[1]-0.42, "$proj_u(x)$", color="#2b6cb0", fontsize=14, weight="bold")
ax.text(lim*u[0]*0.82, lim*u[1]*0.82 - 0.45, "direction $u$", color="#6e6878", fontsize=12)
ax.set_xlim(-0.5, 5.5); ax.set_ylim(-0.5, 4.5); ax.set_aspect("equal")
ax.set_title("Projection onto a direction")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## 4 — Covariance

Before asking which directions matter, the data must be **centered**: subtract the mean of each
column, so that every feature has mean zero.

$$\tilde{X} = X - \bar{x}, \qquad \text{where } \bar{x} \text{ is the vector of column means}$$

> **Centering** shifts the cloud of points so its centre sits at the origin. It changes where the
> data is, not its shape. Every method in this section requires it.

The **covariance matrix** summarises how the features vary together:

$$C = \frac{1}{n-1}\,\tilde{X}^{\top}\tilde{X} \qquad C \in \mathbb{R}^{d \times d}$$

> $\tilde{X}^{\top}$ is the **transpose** of $\tilde{X}$: rows become columns. If $\tilde{X}$ is
> $n \times d$, then $\tilde{X}^{\top}$ is $d \times n$, and the product $\tilde{X}^{\top}\tilde{X}$
> is $d \times d$ — one entry per pair of features.

Reading the entries:

- $C_{jj}$ — the **variance** of feature $j$, its spread on its own
- $C_{jk}$ — the **covariance** between features $j$ and $k$:
    - positive when the two tend to be above their means together
    - negative when one tends to be high while the other is low
    - near zero when knowing one says nothing about the other

Covariance depends on the units of both features, so its magnitude is not comparable across pairs.
Dividing by the two standard deviations gives the **correlation**, which lies between $-1$ and $1$.

The covariance matrix is the object PCA operates on. It contains everything about the shape of the
data cloud, and nothing about its location — which is what centering removed.

---
## 5 — Principal components

Section 3 left a question: which direction $u$ should we project onto?

**The first principal component is the direction along which the projected data has the largest
variance.** Formally, choose the unit vector $u$ maximising

$$\operatorname{Var}(\tilde{X}u) = u^{\top} C\, u \qquad \text{subject to } \|u\| = 1$$

The solution is the **eigenvector** of $C$ with the largest **eigenvalue**.

> A vector $u$ is an **eigenvector** of a matrix $C$ if multiplying by $C$ only rescales it:
> $$C u = \lambda u$$
> The number $\lambda$ is the **eigenvalue**. Multiplying by $C$ does not rotate an eigenvector; it
> only stretches or shrinks it, by a factor of $\lambda$.

For a covariance matrix, the eigenvectors and eigenvalues have a direct reading:

- the **eigenvectors** are the axes of the data cloud, ordered from widest to narrowest
- each **eigenvalue** is the variance of the data along its eigenvector
- the eigenvectors are mutually perpendicular, so they form a new set of coordinate axes

The **principal components** are these eigenvectors. Projecting the data onto the first $k$ of them
gives the best $k$-dimensional summary available, in the sense of retaining the most variance.

The fraction of total variance retained by component $j$ is

$$\frac{\lambda_j}{\lambda_1 + \lambda_2 + \cdots + \lambda_d}$$

Plotting these fractions in order gives a **scree plot**, which is how you decide how many components
to keep.

In [ ]:
# A 2-D cloud with correlated features, so the principal axes are visible.
rng = np.random.default_rng(4)
cloud = rng.multivariate_normal([0, 0], [[3.0, 2.1], [2.1, 2.0]], size=400)

centered = cloud - cloud.mean(axis=0)
C = (centered.T @ centered) / (len(centered) - 1)

evals, evecs = np.linalg.eigh(C)          # eigh: for symmetric matrices; returns ascending
order  = np.argsort(evals)[::-1]          # sort descending
evals, evecs = evals[order], evecs[:, order]

# A component and its negation describe the same axis, so the sign is arbitrary.
# Fix it by convention: make the largest-magnitude entry of each component positive.
for k in range(evecs.shape[1]):
    if evecs[np.argmax(np.abs(evecs[:, k])), k] < 0:
        evecs[:, k] *= -1

print("covariance matrix:\n", np.round(C, 3))
print("\neigenvalues (variance along each axis):", np.round(evals, 3))
print("fraction of total variance:", np.round(evals / evals.sum(), 3))
print("\nfirst principal component :", np.round(evecs[:, 0], 3))
print("second principal component:", np.round(evecs[:, 1], 3))
print("dot product of the two    :", f"{evecs[:,0] @ evecs[:,1]:.2e}   <- perpendicular")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
axes[0].scatter(*centered.T, s=12, alpha=0.45, color="#6e6878")
for k, col in [(0, "#c1272d"), (1, "#2b6cb0")]:
    d = evecs[:, k] * 2.4 * np.sqrt(evals[k])
    axes[0].annotate("", xy=d, xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", lw=3, color=col))
    axes[0].text(d[0]*1.12, d[1]*1.12, f"PC{k+1}", color=col, fontsize=14, weight="bold")
axes[0].set_title("The data, with its principal axes")

rotated = centered @ evecs                 # express every point in the PC coordinate system
axes[1].scatter(*rotated.T, s=12, alpha=0.45, color="#6e6878")
axes[1].axhline(0, color="#2b6cb0", lw=1.4); axes[1].axvline(0, color="#c1272d", lw=1.4)
axes[1].set_xlabel("PC1"); axes[1].set_ylabel("PC2")
axes[1].set_title("The same data, in PC coordinates")
for ax in axes:                      # identical scales, so the rotation is the only difference
    ax.set_xlim(-7.5, 7.5); ax.set_ylim(-5, 5); ax.set_aspect("equal"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print(f"\nvariance along PC1 = {rotated[:,0].var(ddof=1):.3f}   (eigenvalue {evals[0]:.3f})")
print(f"variance along PC2 = {rotated[:,1].var(ddof=1):.3f}   (eigenvalue {evals[1]:.3f})")

The right-hand panel is the same cloud, rotated so the principal components are the axes. Nothing was
discarded — the two panels contain identical information. Rotating first is what makes discarding
cheap: dropping PC2 from the right-hand panel costs little, while dropping either original feature
from the left-hand panel would cost a great deal.

---
## 6 — The singular value decomposition

Forming $C = \frac{1}{n-1}\tilde{X}^\top \tilde{X}$ and then finding its eigenvectors works, but it
is not how PCA is computed in practice. For $d = 20{,}000$ genes, $C$ is a $20{,}000 \times 20{,}000$
matrix — expensive to build, and squaring the data loses numerical precision.

The **singular value decomposition** factors the centered data matrix directly:

$$\tilde{X} = U S V^{\top}$$

where, for $\tilde{X}$ of size $n \times d$:

- $V$ is $d \times k$, and its **columns are the principal components** — the same eigenvectors as
  section 5
- $S$ is diagonal, holding the **singular values** $s_1 \ge s_2 \ge \cdots \ge 0$, which relate to the
  eigenvalues by $\lambda_j = s_j^2 / (n-1)$
- $U$ is $n \times k$, and $US$ holds the **coordinates of each sample** in the new system

> **Diagonal** means every entry off the main diagonal is zero.
>
> **Orthonormal** describes $U$ and $V$: their columns are unit vectors, mutually perpendicular. Such
> a matrix rotates without stretching.

Every decomposition in this course routes through the SVD, and `sklearn.decomposition.PCA` calls it
internally. Two properties are worth carrying forward:

- The SVD gives the best rank-$k$ approximation of $\tilde{X}$ for every $k$ at once — keep the first
  $k$ columns and you have the closest possible $k$-dimensional description of the data.
- It never forms $C$, so it works when $d \gg n$, which is the normal situation in genomics.

In [ ]:
# The SVD reproduces section 5 without ever forming the covariance matrix.
U, S, Vt = np.linalg.svd(centered, full_matrices=False)

print("singular values          :", np.round(S, 3))
print("s^2/(n-1)                :", np.round(S**2 / (len(centered) - 1), 3))
print("eigenvalues from before  :", np.round(evals, 3), "  <- identical\n")

print("first component from SVD :", np.round(Vt[0], 3))
print("first component from eigh:", np.round(evecs[:, 0], 3))
print("\n(a component and its negation describe the same axis, so signs may differ)")

---
## 7 — Embeddings

An **embedding** is a representation of an object as a vector of numbers, chosen so that geometry in
that space carries meaning: similar objects end up close together.

> An **embedding** maps something that is not naturally numeric — a word, an image, a patient record,
> a molecule — into $\mathbb{R}^k$, so it can be measured, compared, and fed to a model.

Section 5 produced one. Projecting a 400-gene tumour profile onto two principal components embeds
each tumour in $\mathbb{R}^2$, and in Meeting 5 you will see that tumours of the same type land near
one another there.

PCA is a **linear** embedding: every coordinate is a weighted sum of the original features, so
$\text{PC}_1$ can be inspected directly to see which genes contribute to it. The embeddings from
pretrained neural networks in Meeting 24 are nonlinear and far more expressive, but they are used the
same way — the distances and dot products of sections 2 and 3 remain the tools for comparing them.

In [ ]:
import pandas as pd

URL = "https://raw.githubusercontent.com/dmochow/BME-i9400-2026/main/data/tcga_rnaseq_subset.csv"
df  = pd.read_csv(URL)

y = df["cancer_type"].values
X = df.drop(columns="cancer_type").values
print(f"X is {X.shape[0]} tumour samples x {X.shape[1]} genes")
print("cancer types:", ", ".join(f"{c} ({(y==c).sum()})" for c in sorted(set(y))))

Xc = X - X.mean(axis=0)
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
frac = S**2 / (S**2).sum()
Z = Xc @ Vt[:2].T                       # embed every tumour in two dimensions

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
axes[0].bar(np.arange(1, 21), 100*frac[:20], color="#5b3a8e")
axes[0].set_xlabel("component"); axes[0].set_ylabel("variance explained (%)")
axes[0].set_title("Scree plot: first 20 of 400 components")
axes[0].set_xticks([1, 5, 10, 15, 20])

for c, col in zip(sorted(set(y)), ["#c1272d", "#2b6cb0", "#e8a33d", "#2f8f5b", "#7d5ba6"]):
    m = y == c
    axes[1].scatter(Z[m, 0], Z[m, 1], s=16, alpha=0.75, color=col, label=c)
axes[1].set_xlabel(f"PC1  ({100*frac[0]:.1f}% of variance)")
axes[1].set_ylabel(f"PC2  ({100*frac[1]:.1f}%)")
axes[1].set_title("801 tumours, 400 genes, drawn in two dimensions")
axes[1].legend(title="cancer type", fontsize=10)
for ax in axes: ax.grid(alpha=0.25)
plt.tight_layout(); plt.show()

print(f"\nPC1 and PC2 together retain {100*frac[:2].sum():.1f}% of the total variance.")

Each point is one tumour, originally described by 400 numbers and here drawn using two. The cancer
types separate, and **no label was used to compute the components** — PCA saw only the expression
matrix. The grouping is a property of the data.

Section 5 of Meeting 5 quantifies what the compression costs: a nearest-neighbour classifier reaches
99.8% accuracy on all 400 genes and 97.9% on these two components.

---
## 8 — Connection to the course

**Every model in this course is built from the operations in sections 2 and 3.** Linear and logistic
regression compute $x \cdot w$ — a projection of the sample onto a weight vector. A neural network
layer computes many dot products at once and applies a nonlinearity. Attention, in Meeting 23, scores
the similarity of two vectors with a dot product.

**Centering and scaling are preprocessing steps with consequences.** PCA is defined on centered data;
features measured on larger scales dominate the covariance matrix unless standardised. In Meeting 11
these become a source of data leakage, since the mean and standard deviation must be computed on the
training set alone.

**Dimensionality reduction is a tool with a cost.** Retaining 45.8% of the variance discarded the
rest, and section 7 showed the discarded part was largely irrelevant to cancer type. That is not
guaranteed — PCA maximises variance, and variance is not the same as usefulness. A direction with
small variance can still be the one that predicts the outcome.

## What to take away

- $X \in \mathbb{R}^{n \times d}$: rows are samples, columns are features
- $\|x - z\|$ measures distance; $x \cdot z$ measures alignment; $\cos\theta$ measures alignment
  independently of magnitude
- Projection onto a unit vector $u$ is the single number $x \cdot u$
- The covariance matrix $C = \frac{1}{n-1}\tilde{X}^\top\tilde{X}$ describes the shape of the centered
  data cloud
- Principal components are the eigenvectors of $C$, ordered by eigenvalue, and are computed in
  practice by the SVD of $\tilde{X}$
- An embedding is a vector representation in which distance carries meaning

### Wednesday, September 16 — Studio

You will implement PCA on the gene expression matrix from section 7: centering, the SVD, the scree
plot, the projection, and a measurement of what the compression costs a classifier.